# Exercícios — Classificação

Soluções recolhidas em `# @title`.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Exercício 1 — Escolher o k do k-NN

In [ ]:
# @title Solução
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

w = load_wine()
melhor, melhor_ac = None, 0
for k in range(1, 30, 2):
    ac = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), w.data, w.target, cv=5).mean()
    if ac > melhor_ac: melhor, melhor_ac = k, ac
print("melhor k:", melhor, "| acuracia CV:", round(melhor_ac, 3))

## Exercício 2 — Ler uma matriz de confusão

In [ ]:
# @title Solução
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

bc = load_breast_cancer()
y = 1 - bc.target   # 1 = maligno
X_tr, X_te, y_tr, y_te = train_test_split(bc.data, y, test_size=0.3, random_state=SEMENTE, stratify=y)
esc = StandardScaler().fit(X_tr)
modelo = LogisticRegression(max_iter=5000).fit(esc.transform(X_tr), y_tr)
proba = modelo.predict_proba(esc.transform(X_te))[:, 1]
previsto = (proba > 0.5).astype(int)
vn, fp, fn, vp = confusion_matrix(y_te, previsto).ravel()
print("VP", vp, "FP", fp, "FN", fn, "VN", vn)
print("precisao:", round(vp/(vp+fp), 3), "| recall:", round(vp/(vp+fn), 3))
print("num rastreio, o falso negativo (maligno dito benigno) e o mais grave.")

## Exercício 3 — Ajustar o limiar de decisão

In [ ]:
# @title Solução
for limiar in [0.5, 0.3, 0.2, 0.1]:
    prev = (proba > limiar).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_te, prev).ravel()
    rec = vp/(vp+fn); prec = vp/(vp+fp) if (vp+fp) else 0
    print("limiar", limiar, "-> recall", round(rec, 3), "| precisao", round(prec, 3))
print("baixar o limiar sobe o recall (pega mais malignos) e baixa a precisao.")

## Exercício 4 — Linear × não linear (o valor do kernel)

In [ ]:
# @title Solução
from sklearn.datasets import make_circles
from sklearn.svm import SVC

Xc, yc = make_circles(n_samples=300, factor=0.4, noise=0.12, random_state=SEMENTE)
for nome, m in [("logistica", LogisticRegression()),
                ("SVM linear", SVC(kernel="linear")),
                ("SVM RBF", SVC(kernel="rbf", gamma=1.0))]:
    ac = cross_val_score(m, Xc, yc, cv=5).mean()
    print(nome.ljust(11), "acuracia CV:", round(ac, 3))
print("so a RBF resolve: o kernel torna o anel separavel.")